In [1]:
%pip install \
    "qiskit==2.2.3" \
    "qiskit-machine-learning==0.9.0" \
    "qiskit-optimization==0.7.0" \
    "qiskit-algorithms==0.4.0" \
    "packaging>=24.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 11.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 20.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.7 MB/s  0:00:00
  Created wheel for docplex: filename=docplex-2.32.264-py3-none-any.whl size=703859 sha256=f83cc776bd5f4c507b5d3285564e56b0b1154d8511ff75d842b77e453eb9b112
  Stored in directory: /root/.cache/pip/wheels/ba/11/de/5d0051be010658975fb2155006b4b8a818136d9f69ed20b4ba
Successfully built docplex
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [qiskit-algorithms]


In [8]:
import numpy as np
from importlib.metadata import version
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print('Qiskit:', version('qiskit'))
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import PauliEvolutionGate

data_value, time_value = 0.8, 1.0
H = SparsePauliOp.from_list([('Z', data_value)])
qc = QuantumCircuit(1)
qc.h(0)
qc.append(PauliEvolutionGate(H, time=time_value), [0])
state = Statevector.from_instruction(qc)
X = SparsePauliOp.from_list([('X', 1.0)])
expectation = float(state.expectation_value(X).real)
print('상태 진폭:', np.round(state.data, 6))
print('X 기대값:', round(expectation, 6))
assert np.isclose(expectation, np.cos(2 * data_value * time_value))

Qiskit: 2.2.3
상태 진폭: [0.492646-0.507247j 0.492646+0.507247j]
X 기대값: -0.0292


X 기대값은 -1~1 범위입니다. 특징 값이 위상에 들어가므로 관측량 선택에 따라 읽어내는 정보가 달라집니다. 여기서는 데이터 의존 계수의 시간 발전을 구현합니다.

In [9]:
import numpy as np
from importlib.metadata import version
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print('Qiskit:', version('qiskit'))
from qiskit.circuit.library import zz_feature_map

data = np.array([0.2, 0.6, 0.8])
other = np.array([0.3, 0.5, 0.7])
feature_map = zz_feature_map(feature_dimension=3, reps=2)
parameters = list(feature_map.parameters)
qc = feature_map.assign_parameters(dict(zip(parameters, data)))
qc_other = feature_map.assign_parameters(dict(zip(parameters, other)))
state = Statevector.from_instruction(qc)
state_other = Statevector.from_instruction(qc_other)
kernel = float(abs(np.vdot(state.data, state_other.data))**2)
print('파라미터 순서:', [str(p) for p in parameters])
print('남은 미지정 파라미터:', qc.num_parameters)
print('K(x,x):', round(float(abs(np.vdot(state.data, state.data))**2), 6))
print('K(x,other):', round(kernel, 6))
assert qc.num_parameters == 0 and 0 <= kernel <= 1 + 1e-10

Qiskit: 2.2.3
파라미터 순서: ['x[0]', 'x[1]', 'x[2]']
남은 미지정 파라미터: 0
K(x,x): 1.0
K(x,other): 0.521441


미지정 파라미터가 0개인지 확인합니다. 다른 입력의 커널 값과 자기 유사도를 함께 출력하여 데이터가 실제 회로에 들어갔는지 점검합니다.


In [10]:
import numpy as np
from importlib.metadata import version
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print('Qiskit:', version('qiskit'))
data = np.array([0.2, 0.4, 0.3])
amplitudes = data / np.linalg.norm(data)
n = len(data)
unary = np.zeros(2**n, dtype=complex)
positions = 1 << np.arange(n)
unary[positions] = amplitudes
qc = QuantumCircuit(n)
qc.initialize(unary, list(range(n)))
state = Statevector.from_instruction(qc)
print('활성 기저상태:', [format(int(k), f'0{n}b') for k in positions])
print('진폭:', np.round(state.data[positions], 6))
print('측정 확률:', state.probabilities_dict())
assert np.allclose(state.data[positions], amplitudes)
assert np.isclose(state.probabilities()[positions].sum(), 1.0)

Qiskit: 2.2.3
활성 기저상태: ['001', '010', '100']
진폭: [0.371391+0.j 0.742781+0.j 0.557086+0.j]
측정 확률: {np.str_('001'): np.float64(0.13793103448275856), np.str_('010'): np.float64(0.5517241379310343), np.str_('100'): np.float64(0.31034482758620674)}


확률은 1이 한 개인 세 상태에만 존재합니다. 단일 기저상태 하나만 선택하는 경우와 여러 단일 여기 상태에 진폭을 배분하는 경우를 입력 값으로 확인할 수 있습니다.